In [1]:
!pip install transformers torch datasets sentence-transformers -q

In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU available: True
Device: Tesla T4


In [3]:


import torch
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Transformers version:", __import__('transformers').__version__)
print("PyTorch version:", torch.__version__)

Transformers version: 5.0.0
PyTorch version: 2.10.0+cu128


In [4]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Test sentences — including ones Day 8 model got wrong
sentences = [
    "This movie was great and I enjoyed it",
    "Complete garbage not worth watching",
    "The cinematography was stunning but the story was weak",
    "Absolutely brilliant and moving performance",
    "I did not like this film at all"
]

print("=== Sentiment Analysis ===")
for sentence in sentences:
    result = sentiment_pipeline(sentence)[0]
    print(f"Text:       {sentence}")
    print(f"Sentiment:  {result['label']}, Confidence: {result['score']:.2%}")
    print()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

=== Sentiment Analysis ===
Text:       This movie was great and I enjoyed it
Sentiment:  POSITIVE, Confidence: 99.99%

Text:       Complete garbage not worth watching
Sentiment:  NEGATIVE, Confidence: 99.98%

Text:       The cinematography was stunning but the story was weak
Sentiment:  NEGATIVE, Confidence: 99.71%

Text:       Absolutely brilliant and moving performance
Sentiment:  POSITIVE, Confidence: 99.99%

Text:       I did not like this film at all
Sentiment:  NEGATIVE, Confidence: 99.07%



In [5]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

text = "Transformers revolutionized NLP with self-attention mechanisms"

# Tokenize
tokens = tokenizer.tokenize(text)
print("Original text:", text)
print("Word count:", len(text.split()))
print("Token count:", len(tokens))
print("Tokens:", tokens)

# Full encoding
encoding = tokenizer(text, return_tensors="pt")
print("\nInput IDs:", encoding['input_ids'])
print("Attention Mask:", encoding['attention_mask'])

# Decode back
decoded = tokenizer.decode(encoding['input_ids'][0])
print("\nDecoded:", decoded)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Original text: Transformers revolutionized NLP with self-attention mechanisms
Word count: 6
Token count: 10
Tokens: ['transformers', 'revolution', '##ized', 'nl', '##p', 'with', 'self', '-', 'attention', 'mechanisms']

Input IDs: tensor([[  101, 19081,  4329,  3550, 17953,  2361,  2007,  2969,  1011,  3086,
         10595,   102]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Decoded: [CLS] transformers revolutionized nlp with self - attention mechanisms [SEP]


In [6]:
# Task 1 — Zero Shot Classification
print("=== Zero Shot Classification ===")
zero_shot = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli"
)

text = "The new iPhone has an amazing camera and long battery life"
labels = ["technology", "sports", "politics", "entertainment"]

result = zero_shot(text, candidate_labels=labels)
print(f"Text: {text}")
print("Scores:")
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label:<15} {score:.2%}")

print()

# Task 2 — Named Entity Recognition
print("=== Named Entity Recognition ===")
ner = pipeline(
    "ner",
    model="elastic/distilbert-base-uncased-finetuned-conll03-english",
    aggregation_strategy="simple"
)

text = "Elon Musk founded SpaceX in 2002 in California"
print(f"Text: {text}")
print("Entities found:")
entities = ner(text)
for entity in entities:
    print(f"  {entity['word']:<15} → {entity['entity_group']} ({entity['score']:.2%})")

=== Zero Shot Classification ===


config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Text: Elon Musk founded SpaceX in 2002 in California
Entities found:
  el              → PER (85.00%)
  ##on musk       → PER (77.34%)
  space           → ORG (99.80%)
  ##x             → ORG (99.82%)
  california      → LOC (99.91%)


In [7]:
from transformers import AutoTokenizer, AutoModel

model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
embed_model = AutoModel.from_pretrained(model_name)

def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = embed_model(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze().numpy()

# Test sentences
sentences = [
    "I love programming in Python",
    "Python is my favorite coding language",
    "I enjoy playing football",
    "Soccer is a great sport",
    "Machine learning is fascinating"
]

embeddings = [get_embedding(s) for s in sentences]
print(f"Embedding shape: {embeddings[0].shape}")
print(f"Each sentence → {embeddings[0].shape[0]}-dimensional vector")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (384,)
Each sentence → 384-dimensional vector


In [8]:
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

pairs = [
    (0, 1),  # Python vs Python
    (2, 3),  # Football vs Soccer
    (0, 2),  # Python vs Football
    (0, 4),  # Python vs ML
]

print("Similarity Scores:")
print(f"{'Sentence 1':<35} {'Sentence 2':<35} {'Score'}")
print("-" * 80)
for i, j in pairs:
    sim = cosine_similarity(embeddings[i], embeddings[j])
    print(f"{sentences[i][:33]:<35} {sentences[j][:33]:<35} {sim:.4f}")

Similarity Scores:
Sentence 1                          Sentence 2                          Score
--------------------------------------------------------------------------------
I love programming in Python        Python is my favorite coding lang   0.8571
I enjoy playing football            Soccer is a great sport             0.5975
I love programming in Python        I enjoy playing football            0.2862
I love programming in Python        Machine learning is fascinating     0.3141


In [9]:
documents = [
    "Python is a high-level programming language known for simplicity",
    "Machine learning algorithms learn patterns from data automatically",
    "Neural networks are inspired by the human brain structure",
    "Deep learning uses multiple layers to learn complex representations",
    "Natural language processing enables computers to understand text",
    "Computer vision allows machines to interpret visual information",
    "Reinforcement learning trains agents through rewards and penalties",
    "Transfer learning reuses knowledge from one task to another",
    "Gradient descent optimizes model parameters to minimize loss",
    "Transformers use attention mechanisms to process sequential data"
]

print("Embedding documents...")
doc_embeddings = [get_embedding(doc) for doc in documents]
print(f"Done. Embedded {len(documents)} documents.")

def semantic_search(query, documents, doc_embeddings, top_k=3):
    query_embedding = get_embedding(query)
    similarities = [
        cosine_similarity(query_embedding, doc_emb)
        for doc_emb in doc_embeddings
    ]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    print(f"\nQuery: '{query}'")
    print(f"Top {top_k} results:")
    for rank, idx in enumerate(top_indices, 1):
        print(f"  {rank}. [{similarities[idx]:.4f}] {documents[idx]}")

# Run searches
semantic_search("how do machines learn from examples", documents, doc_embeddings)
semantic_search("what is used to understand human language", documents, doc_embeddings)
semantic_search("how are model weights updated during training", documents, doc_embeddings)

Embedding documents...
Done. Embedded 10 documents.

Query: 'how do machines learn from examples'
Top 3 results:
  1. [0.5661] Machine learning algorithms learn patterns from data automatically
  2. [0.4826] Computer vision allows machines to interpret visual information
  3. [0.4356] Natural language processing enables computers to understand text

Query: 'what is used to understand human language'
Top 3 results:
  1. [0.5319] Natural language processing enables computers to understand text
  2. [0.3317] Computer vision allows machines to interpret visual information
  3. [0.2922] Python is a high-level programming language known for simplicity

Query: 'how are model weights updated during training'
Top 3 results:
  1. [0.3219] Gradient descent optimizes model parameters to minimize loss
  2. [0.2931] Deep learning uses multiple layers to learn complex representations
  3. [0.2919] Transfer learning reuses knowledge from one task to another
